In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, os

BASE = "/content/drive/MyDrive/Noshows"
DATA, REPORTS = BASE + "/data", BASE + "/reports"
os.makedirs(REPORTS, exist_ok=True)
assert os.path.exists(DATA + "/data_clean.csv"), "data_clean.csv is missing in data/"

Mounted at /content/drive


Load and prepare

In [ ]:
df = pd.read_csv(DATA + "/data_clean.csv")
df.columns = df.columns.str.strip()
df["Jour programmé"] = pd.to_datetime(df["Jour programmé"])
df["Jour de rendez-vous"] = pd.to_datetime(df["Jour de rendez-vous"])

df["no_show"] = (df["Non-présentation"] == "Yes").astype(int)
df["WaitingDays"] = (df["Jour de rendez-vous"].dt.normalize()
                     - df["Jour programmé"].dt.normalize()).dt.days
df["Weekday"] = df["Jour de rendez-vous"].dt.dayofweek

print("Rows:", len(df), "| no-show rate:", round(df["no_show"].mean() * 100, 1), "%")

Rows: 110515 | no-show rate: 20.2 %


Summary statistics of the numeric variables

In [ ]:
num_cols = ["Age", "WaitingDays", "Weekday", "Bourse", "SMS_reçu"]
desc = df[num_cols].agg(["count", "mean", "median", "std", "min", "max"]).T.round(2)
desc.index.name = "variable"
desc

,count,mean,median,std,min,max
variable,,,,,,
Age,110515.0,37.08,37.0,23.10,0.0,100.0
WaitingDays,110515.0,10.18,4.0,15.26,0.0,179.0
Weekday,110515.0,1.86,2.0,1.37,0.0,5.0
Bourse,110515.0,0.10,0.0,0.30,0.0,1.0
SMS_reçu,110515.0,0.32,0.0,0.47,0.0,1.0


No-show rate for each group

In [ ]:
rows = []
for col in ["Genre", "SMS_reçu", "Bourse", "Radio dentaire réalisée", "Weekday", "Nom du dentiste"]:
    g = df.groupby(col)["no_show"].agg(["count", "mean"])
    for value, r in g.iterrows():
        rows.append({"variable": col, "value": value,
                     "patients": int(r["count"]),
                     "no_show_rate_%": round(r["mean"] * 100, 1)})
rates = pd.DataFrame(rows)
rates

,variable,value,patients,no_show_rate_%
0,Genre,F,71830,20.3
1,Genre,M,38685,20.0
2,SMS_reçu,0,75034,16.7
3,SMS_reçu,1,35481,27.6
4,Bourse,0,99654,19.8
5,Bourse,1,10861,23.7
6,Radio dentaire réalisée,Non,55056,19.9
7,Radio dentaire réalisée,Oui,55459,20.5
8,Weekday,0,22712,20.6
9,Weekday,1,25637,20.1


Which variables are linked to no-shows (correlation)

In [ ]:
corr = (df[num_cols + ["no_show"]].corr()["no_show"]
        .drop("no_show").round(3).sort_values(key=abs, ascending=False))
corr

,no_show
WaitingDays,0.186
SMS_reçu,0.127
Age,-0.060
Bourse,0.029
Weekday,0.001


Is a difference real or just chance? (optional, for the report)

In [ ]:
from scipy.stats import chi2_contingency, ttest_ind

for col in ["SMS_reçu", "Bourse", "Genre"]:
    table = pd.crosstab(df[col], df["no_show"])
    chi, p, _, _ = chi2_contingency(table)
    print(col, "p-value =", round(p, 4))

a = df.loc[df["no_show"] == 1, "WaitingDays"]
b = df.loc[df["no_show"] == 0, "WaitingDays"]
print("WaitingDays p-value =", round(ttest_ind(a, b, equal_var=False).pvalue, 4))

SMS_reçu p-value = 0.0
Bourse p-value = 0.0
Genre p-value = 0.1751
WaitingDays p-value = 0.0


Save for the team

In [ ]:
desc.to_csv(REPORTS + "/statistiques.csv")
rates.to_csv(REPORTS + "/taux_noshow_par_groupe.csv", index=False)
print(os.listdir(REPORTS))

['figures', 'comparaison_algorithmes.csv', 'statistiques.csv', 'taux_noshow_par_groupe.csv']
